# 00 · Reference system setup (Colab)
**Advanced GenAI HS26 · Trustworthy Agentic RAG · Group: Divya, Farah, Hanieh**

Runs Cyriel & Vincent's reference implementation on the **shared `full_corpus` artifacts**, answers a few test queries, and saves the decision traces. Any setup or compatibility problem is logged automatically to `setup_issues.md` (for Guang).

### One-time prerequisites (each group member)
1. **Shortcut to the shared artifacts.** Open the [RAG_Project_Artifact folder](https://drive.google.com/drive/folders/1drtdc_QcLHu5mcfEB9GWyWA67hoxws45) → right-click → *Organize → Add shortcut* → **My Drive**. (Colab can only see folders that are in My Drive.)
2. **Reference code.** Put `Baseline.zip` (from the course materials) in a folder in My Drive, e.g. `My Drive/AdvGenAI/Baseline.zip`. Use the zip rather than GitHub: the GitHub repo leaves out the qrels (`benchmark/score/`), which the retrieval metrics need.
3. **API key.** Colab sidebar → 🔑 *Secrets* → add `ANTHROPIC_API_KEY` and switch on *Notebook access*. Never paste the key into a cell.
4. **GPU.** *Runtime → Change runtime type → T4 GPU*.

Then run all the cells in order. The first run downloads about 3 GB of HuggingFace models (e5-large, M2M100, FLAN-T5, MiniLM).

## 1 · Config

In [ ]:
# ---- edit if your Drive layout differs --------------------------------------
WORK_DIR_NAME   = "AdvGenAI"                 # our group folder in My Drive (outputs go here)
ARTIFACT_FOLDER = "RAG_Project_Artifact"     # name of the shortcut to the shared folder
COPY_TO_LOCAL   = True    # copy indexes to Colab disk: faster, and avoids Chroma writing to a read-only folder

# Models: the reference run used these. Keep them for reproduction; a cheaper
# generator model is fine for exploratory runs, but note it in the log.
AGENT_MODEL_GENERATION = "claude-opus-4-7"
AGENT_MODEL_VERIFIER   = "claude-opus-4-7"
AGENT_MODEL_UTILITY    = "claude-haiku-4-5"

## 2 · Issue log + environment check

In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, pathlib, platform, datetime
ISSUES = []
def log_issue(step, err=None, note=""):
    tb = traceback.format_exc() if err is not None else ""
    ISSUES.append({"time": datetime.datetime.now().isoformat(timespec="seconds"),
                   "step": step, "error": repr(err) if err else "", "note": note, "traceback": tb[-3000:]})
    print(f"⚠️  logged issue at [{step}]: {err or note}")

def step(name):
    # Use as: with step("x"): ...  → exceptions are logged, not raised
    class _S:
        def __enter__(self): print(f"\n▶ {name}"); self.t = time.time()
        def __exit__(self, et, ev, tb):
            if ev is not None:
                log_issue(name, ev); return True
            print(f"  ✓ {name} ({time.time()-self.t:.0f}s)")
    return _S()

print("Python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
    if not torch.cuda.is_available():
        log_issue("gpu", note="No GPU runtime: model loading/retrieval will be very slow on CPU.")
except Exception as e:
    log_issue("torch import", e)

## 3 · Mount Drive and locate the artifacts and code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = pathlib.Path("/content/drive/MyDrive")
WORK = MYDRIVE / WORK_DIR_NAME
WORK.mkdir(parents=True, exist_ok=True)

def find_one(pattern, roots, maxdepth=4):
    for root in roots:
        for depth in range(maxdepth + 1):
            hits = list(root.glob("/".join(["*"] * depth + [pattern])) if depth else root.glob(pattern))
            if hits: return hits[0]
    return None

with step("locate shared full_corpus"):
    FULL_CORPUS_SRC = None
    cand = MYDRIVE / ARTIFACT_FOLDER / "storage" / "full_corpus"
    FULL_CORPUS_SRC = cand if cand.is_dir() else find_one("storage/full_corpus", [MYDRIVE], 3)
    assert FULL_CORPUS_SRC, ("full_corpus not found. Did you add the RAG_Project_Artifact shortcut to My Drive?")
    print("  full_corpus:", FULL_CORPUS_SRC)
    for sub in ["retrieval/fixed_size_chunk/bm25_retriever_full.pkl", "vectordb_dense/fixed_e5/chroma.sqlite3",
                "retrieval_graph/comm2chunk_fixed.json", "retrieval_graph/embeddings", "Lang_norm/fixed_size_chunk"]:
        p = FULL_CORPUS_SRC / sub
        print(f"   {'OK ' if p.exists() else '!! '} {sub}")
        if not p.exists(): log_issue("artifact check", note=f"missing in shared folder: {sub}")

with step("locate Baseline.zip"):
    ZIP = WORK / "Baseline.zip"
    ZIP = ZIP if ZIP.exists() else find_one("Baseline.zip", [MYDRIVE], 3)
    print("  Baseline.zip:", ZIP)

## 4 · Unpack the code and stage the indexes

In [ ]:
REPO = pathlib.Path("/content/Baseline")          # repo root (contains src/, results/, Baseline/benchmark)
with step("unpack reference code"):
    if not (REPO / "src").exists():
        if ZIP:
            subprocess.run(["unzip", "-q", "-o", str(ZIP), "-d", "/content"], check=True)
        else:
            log_issue("code source", note="Baseline.zip not found, so cloning GitHub (no qrels: retrieval metrics unavailable)")
            subprocess.run(["git", "clone", "-q", "https://github.com/CyrielVanHelleputte/Adv.-Gen-AI", str(REPO)], check=True)
    assert (REPO / "src" / "agentic" / "pipeline.py").exists(), "unexpected repo layout"
    print("  repo:", REPO)

BASELINE_ROOT = REPO / "Baseline"                 # what config.py expects
STORAGE_LINK  = BASELINE_ROOT / "storage" / "full_corpus"
with step("stage full_corpus"):
    STORAGE_LINK.parent.mkdir(parents=True, exist_ok=True)
    if STORAGE_LINK.is_symlink() or STORAGE_LINK.exists():
        print("  already staged:", STORAGE_LINK.resolve())
    elif COPY_TO_LOCAL:
        local = pathlib.Path("/content/full_corpus")
        if not local.exists():
            print("  copying shared indexes to local disk (a few minutes)...")
            subprocess.run(["rsync", "-a", "--exclude", "__pycache__", f"{FULL_CORPUS_SRC}/", f"{local}/"], check=True)
        STORAGE_LINK.symlink_to(local)
    else:
        STORAGE_LINK.symlink_to(FULL_CORPUS_SRC)
    print("  du:", subprocess.run(["du", "-sh", str(STORAGE_LINK.resolve())], capture_output=True, text=True).stdout.strip())

## 5 · Install dependencies
The `langchain` 0.3 pins keep the BM25 pickle loadable. Unpinned versions changed the pickled classes. `docling`, `spacy` and the other Step 1 parsing libraries aren't needed here.

In [ ]:
%%capture pip_out
!pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-core>=0.3,<0.4" \
    "langchain-huggingface>=0.1,<0.4" chromadb "rank_bm25>=0.2.2" "sentence-transformers>=3.0" \
    "pytrec_eval-terrier>=0.5.7" langdetect sentencepiece "anthropic>=0.69" python-dotenv

In [ ]:
tail = pip_out.stdout[-1500:] + pip_out.stderr[-1500:]
if "ERROR" in tail: log_issue("pip install", note=tail)
from importlib.metadata import version
VERSIONS = {p: version(p) for p in ["langchain","langchain-community","langchain-core","langchain-huggingface",
            "chromadb","sentence-transformers","transformers","torch","anthropic","numpy"]}
print(json.dumps(VERSIONS, indent=1))

## 6 · API key and paths → environment, then the smoke test (no models)

In [ ]:
from google.colab import userdata
with step("read ANTHROPIC_API_KEY secret"):
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["BASELINE_ROOT"] = str(BASELINE_ROOT)
os.environ["AGENT_MODEL_GENERATION"] = AGENT_MODEL_GENERATION
os.environ["AGENT_MODEL_VERIFIER"]   = AGENT_MODEL_VERIFIER
os.environ["AGENT_MODEL_UTILITY"]    = AGENT_MODEL_UTILITY

os.chdir(REPO)
SRC = str(REPO / "src")
if SRC not in sys.path: sys.path.insert(0, SRC)

with step("smoke test (imports, paths, qa, qrels)"):
    r = subprocess.run([sys.executable, "src/scripts/smoke_test.py"], capture_output=True, text=True, env=os.environ)
    print(r.stdout[-3000:]);
    if r.returncode != 0: raise RuntimeError(f"smoke_test exit {r.returncode}: {r.stderr[-1500:]}")

## 7 · Replay demo (pre-computed traces, no models, no cost)

In [ ]:
with step("replay demo"):
    r = subprocess.run([sys.executable, "-m", "src.scripts.demo"], capture_output=True, text=True)
    print(r.stdout[:6000])
    if r.returncode != 0: raise RuntimeError(r.stderr[-1500:])

## 8 · Load the live system (BM25 + Dense + GraphRAG + agents)
The first run downloads the models. Expect about 5–10 minutes.

In [ ]:
from baseline import config
from agentic import AgenticRAG
rag = None
with step("AgenticRAG.from_pretrained()"):
    print("  device:", config.DEVICE, "| BASELINE_ROOT:", config.BASELINE_ROOT)
    print("  models:", config.AGENT_MODEL_GENERATION, config.AGENT_MODEL_VERIFIER, config.AGENT_MODEL_UTILITY)
    rag = AgenticRAG.from_pretrained()

## 9 · Run a few queries and inspect answers and decision traces

In [ ]:
import textwrap
RUN_DIR = WORK / "runs" / datetime.datetime.now().strftime("%Y%m%d_%H%M")
(RUN_DIR / "traces").mkdir(parents=True, exist_ok=True)
PREV = REPO / "results" / "agentic"     # previous team's committed traces, for reproduction checks

def show_trace(t, width=100):
    f = t["final"]; p = t.get("profile", {})
    print("=" * width)
    print(f"[{t['qid']}] {t['query']}")
    print(f"  profile: lang={p.get('lang')} type={p.get('query_type')} ambiguous={p.get('ambiguity_flag')} entities={p.get('entities')}")
    for s in t["timeline"]:
        k, ms = s["step"], s.get("t_ms", 0)
        if   k == "retrieve": d = f"mode={s['mode']} n={s['n_chunks']} q={s['query'][:60]!r}"
        elif k == "analyze":  d = " ".join(f"{a}={b}" for a, b in s["signals"].items())
        elif k == "decide":   d = f"→ {s['action']}  ({s['reason']})  retries_left={s.get('retries_left')}"
        elif k == "generate": d = f"evidence_found={s.get('evidence_found')} cites={s.get('citations')}"
        elif k == "verify":   d = " ".join(f"{a}={s[a]}" for a in s if a not in ("step","t_ms"))[:160]
        elif k == "recover":  d = f"{s.get('old_mode')}→{s.get('new_mode')} rewritten={s.get('rewritten_query','')[:60]!r}"
        elif k == "final":    continue
        else:                 d = ""
        print(f"  {ms:>7} ms  {k.upper():<9} {d}")
    print(f"  FINAL: {f.get('action','?').upper()}  conf={f.get('confidence')}  mode={f.get('mode_used')}")
    txt = f.get("answer") or f.get("clarifying_question") or f.get("abstain_reason") or ""
    print(textwrap.indent(textwrap.fill(str(txt), width - 4), "    "))

def run_query(qid, query, qa_item=None):
    t0 = time.perf_counter()
    trace = rag.answer(str(qid), query, qa_item).to_dict()
    trace["latency_s"] = round(time.perf_counter() - t0, 2)
    (RUN_DIR / "traces" / f"{qid}.json").write_text(json.dumps(trace, indent=2, ensure_ascii=False), encoding="utf-8")
    show_trace(trace)
    prev = PREV / "traces" / f"{qid}.json"
    if prev.exists():
        pa = json.loads(prev.read_text())["final"].get("action")
        same = "✓ same" if pa == trace["final"].get("action") else "✗ DIFFERENT"
        print(f"  reproduction vs previous team: they={pa}  us={trace['final'].get('action')}  {same}")
    print(f"  latency {trace['latency_s']} s")
    return trace

qa = json.loads((BASELINE_ROOT / "benchmark" / "benchmark_qa_bilingual.json").read_text(encoding="utf-8"))
QA = {str(x["id"]): x for x in qa}
print(len(QA), "benchmark questions loaded")

In [ ]:
# Test set: one expected success, two audit cases, one German query.
#   Q4  InSight on Mars       → previous run: grounded answer
#   Q1  ETH president 2003    → previous run: "correct" abstain. The corpus says Olaf Kübler served until end-2004 (audit case)
#   Q18 Hong Kong connections → previous run: false abstention
TEST = [("4", None), ("1", None), ("18", None),
        ("DE1", "Wer war 2003 Präsident der ETH Zürich?")]
traces = {}
if rag is None:
    print("System not loaded, so skipping. See the issue log.")
else:
    for qid, q in TEST:
        with step(f"query {qid}"):
            item = QA.get(qid)
            traces[qid] = run_query(qid, q or item["question"], item)

## 10 · Optional: first N benchmark questions with the reference script (about $0.04 per question)

In [ ]:
N = 0   # set to e.g. 5, or 25 for the full reproduction (~12 min, ~$1)
if N and rag is not None:
    from scripts.run_agentic import run_benchmark
    with step(f"run_benchmark first {N}"):
        summary = run_benchmark(rag, qa[:N], RUN_DIR / "benchmark", "full")
        print(json.dumps(summary, indent=1))

## 11 · Save the run log and the issue log (for Guang)

In [ ]:
usage = rag.llm.usage.as_dict() if rag is not None else {}
meta = {"run_dir": str(RUN_DIR), "full_corpus_src": str(FULL_CORPUS_SRC), "copied_to_local": COPY_TO_LOCAL,
        "models": {"generation": AGENT_MODEL_GENERATION, "verifier": AGENT_MODEL_VERIFIER, "utility": AGENT_MODEL_UTILITY},
        "versions": globals().get("VERSIONS", {}), "llm_usage": usage, "n_issues": len(ISSUES)}
(RUN_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2))

lines = [f"# Setup / compatibility issues — {datetime.date.today()}", "",
         f"Environment: Colab, Python {platform.python_version()}, GPU={torch.cuda.is_available()}", "",
         "```json", json.dumps(meta['versions'], indent=1), "```", ""]
if not ISSUES: lines.append("No issues: the reference system loaded and ran on the shared artifacts.")
for i, x in enumerate(ISSUES, 1):
    lines += [f"## {i}. {x['step']}", f"- time: {x['time']}", f"- error: `{x['error']}`" if x['error'] else "",
              f"- note: {x['note']}" if x['note'] else "", "", "```", x['traceback'].strip(), "```" if x['traceback'] else "", ""]
(RUN_DIR / "setup_issues.md").write_text("\n".join(l for l in lines if l is not None))
print(f"Saved to {RUN_DIR}:", [p.name for p in RUN_DIR.iterdir()])
print(f"{len(ISSUES)} issue(s) logged | LLM cost so far ≈ ${usage.get('est_cost_usd', 0):.2f}")